# Colab QLoRA: Ministral 3B grounded-response adapter

Runtime → Change runtime type → **T4 GPU**. This notebook makes no Mistral API calls. Upload `synthea_manifest.jsonl` when prompted; it contains the already-created patient-level 80/20 split.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes peft trl datasets

import json, hashlib, random, re, shutil
from pathlib import Path
import numpy as np
import torch
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. Select Runtime → Change runtime type → T4 GPU, then restart.')

MODEL_ID = 'mistralai/Ministral-3-3B-Instruct-2512'
SEED = 42
MAX_SEQUENCE_LENGTH = 512
MAX_TRAIN_PATIENTS = None  # Set 2000 for a smoke test; None uses all 19,840 training patients.
MAX_TEST_PATIENTS = None   # Set 200 for a smoke test; None uses all 5,075 held-out patients.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))
uploaded = files.upload()
if 'synthea_manifest.jsonl' not in uploaded:
    raise FileNotFoundError('Upload the file named synthea_manifest.jsonl.')
MANIFEST_PATH = Path('/content/synthea_manifest.jsonl')

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer

SYSTEM = 'You are a strict medical-record assistant. Answer only from the supplied record excerpt. If the excerpt does not support the answer, respond exactly: Information not found in medical records.'
ABSTAIN = 'Information not found in medical records.'
def example(record):
    lines = [x.strip() for x in record['text'].splitlines() if x.strip()]
    candidates = lines[1:] or lines
    excerpt = candidates[int(hashlib.sha256(record['patient_id'].encode()).hexdigest(), 16) % len(candidates)][:600]
    answerable = int(hashlib.sha256((record['patient_id'] + ':label').encode()).hexdigest(), 16) % 2 == 0
    question = 'What is explicitly documented in this medical-record excerpt?' if answerable else "What is the patient's blood type?"
    answer = f'The record states: {excerpt}' if answerable else ABSTAIN
    return {'patient_id': record['patient_id'], 'user': f'PATIENT RECORD EXCERPT:\n{excerpt}\n\nQUESTION: {question}', 'answer': answer, 'answerable': answerable}

records = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding='utf-8').splitlines()]
train = [example(x) for x in records if x['split'] == 'train'][:MAX_TRAIN_PATIENTS]
test = [example(x) for x in records if x['split'] == 'test'][:MAX_TEST_PATIENTS]
assert not (set(x['patient_id'] for x in train) & set(x['patient_id'] for x in test))
print({'train': len(train), 'test': len(test)})

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
def format_item(item):
    messages = [{'role':'system','content':SYSTEM}, {'role':'user','content':item['user']}, {'role':'assistant','content':item['answer']}]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}
dataset = Dataset.from_list(train).map(format_item, remove_columns=Dataset.from_list(train).column_names)
validation_size = min(1000, max(1, len(dataset)//20))
splits = dataset.train_test_split(test_size=validation_size, seed=SEED)

In [ ]:
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant, device_map='auto')
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj', 'v_proj']))
model.print_trainable_parameters()
args = TrainingArguments(output_dir='/content/ministral-synthea-lora', num_train_epochs=1, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16, learning_rate=2e-4, logging_steps=25, eval_strategy='steps', eval_steps=200, save_strategy='steps', save_steps=200, save_total_limit=2, fp16=True, report_to='none')
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=splits['train'], eval_dataset=splits['test'], dataset_text_field='text', max_seq_length=MAX_SEQUENCE_LENGTH, args=args)
trainer.train()
trainer.save_model('/content/ministral-synthea-lora')
tokenizer.save_pretrained('/content/ministral-synthea-lora')

In [ ]:
# Rebuild the held-out dataset so this evaluation cell can be rerun independently.
if 'example' not in globals():
    def example(record):
        lines = [x.strip() for x in record['text'].splitlines() if x.strip()]
        candidates = lines[1:] or lines
        excerpt = candidates[int(hashlib.sha256(record['patient_id'].encode()).hexdigest(), 16) % len(candidates)][:600]
        answerable = int(hashlib.sha256((record['patient_id'] + ':label').encode()).hexdigest(), 16) % 2 == 0
        question = 'What is explicitly documented in this medical-record excerpt?' if answerable else "What is the patient's blood type?"
        answer = f'The record states: {excerpt}' if answerable else ABSTAIN
        return {'patient_id': record['patient_id'], 'user': f'PATIENT RECORD EXCERPT:\n{excerpt}\n\nQUESTION: {question}', 'answer': answer, 'answerable': answerable}
if 'test' not in globals():
    records = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding='utf-8').splitlines()]
    test = [example(x) for x in records if x['split'] == 'test'][:MAX_TEST_PATIENTS]
print(f'Evaluating {len(test)} held-out patients')

def generate(item):
    messages = [{'role':'system','content':SYSTEM}, {'role':'user','content':item['user']}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        ids = model.generate(**inputs, max_new_tokens=180, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
def norm(x): return re.sub(r'\s+', ' ', x.lower()).strip()
tp=fp=fn=tn=exact=0; predictions=[]
for n, item in enumerate(test, 1):
    prediction = generate(item); predicted_answerable = norm(ABSTAIN) not in norm(prediction)
    if predicted_answerable and item['answerable']: tp += 1
    elif predicted_answerable: fp += 1
    elif item['answerable']: fn += 1
    else: tn += 1
    exact += norm(prediction) == norm(item['answer'])
    predictions.append({**item, 'prediction': prediction})
    if n % 100 == 0: print(f'{n}/{len(test)} evaluated')
metrics = {'test_patients':len(test), 'accuracy':(tp+tn)/len(test), 'precision':tp/(tp+fp) if tp+fp else 0, 'recall':tp/(tp+fn) if tp+fn else 0, 'exact_answer_accuracy':exact/len(test), 'tp':tp, 'fp':fp, 'fn':fn, 'tn':tn, 'lora_rank':8}
Path('/content/metrics.json').write_text(json.dumps(metrics, indent=2)); Path('/content/test_predictions.jsonl').write_text(''.join(json.dumps(x)+'\n' for x in predictions))
shutil.make_archive('/content/ministral-synthea-lora', 'zip', '/content/ministral-synthea-lora')
print(json.dumps(metrics, indent=2))
files.download('/content/ministral-synthea-lora.zip'); files.download('/content/metrics.json'); files.download('/content/test_predictions.jsonl')